# Unity Catalog Demo with retail_demo

This notebook demonstrates key Unity Catalog features using the retail_demo catalog with its medallion architecture (bronze → silver → gold). Covers: namespace discovery, column comments, PII tagging, column masking, access control, and lineage.

In [0]:
%sql
-- Unity Catalog organizes data as catalog.schema.table
-- Browse the hierarchy in the retail_demo catalog
SHOW SCHEMAS IN retail_demo;

-- List tables in each layer
SHOW TABLES IN retail_demo.bronze;
SHOW TABLES IN retail_demo.silver;
SHOW TABLES IN retail_demo.gold;

In [0]:
%sql
-- The information schema provides a governed, queryable view of all metadata
SELECT table_schema, table_name, table_type
FROM retail_demo.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'gold')
ORDER BY table_schema, table_name;

In [0]:
%sql
-- Column comments make data discoverable and self-documenting
-- Adding comments to retail_demo.silver.promotion_sales
COMMENT ON COLUMN retail_demo.silver.promotion_sales.sale_id IS 'Unique identifier for each sale transaction';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.sale_date IS 'Date when the sale occurred';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.store_id IS 'Identifier for the store where the sale took place';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.store_name IS 'Human-readable store name';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.product_id IS 'Identifier for the product sold';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.product_name IS 'Human-readable product name';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.category IS 'Product category (e.g., Electronics, Grocery)';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.regular_price IS 'Standard price before any promotion discount';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.gross_margin_pct IS 'Gross margin percentage (0-100)';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.promotion_id IS 'Identifier for the promotion applied, NULL if no promotion';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.discount_pct IS 'Percentage discount applied by the promotion (0-100)';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.promotion_name IS 'Human-readable promotion name';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.units_sold IS 'Number of units sold in this transaction';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.revenue IS 'Total revenue for this sale after promotion discount';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.discount_amount IS 'Absolute discount amount in currency';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.promotion IS 'Flag indicating whether a promotion was applied';
COMMENT ON COLUMN retail_demo.silver.promotion_sales.silver_ingested_at IS 'Timestamp when this row was loaded into the silver layer';

-- Verify the comments
DESCRIBE TABLE EXTENDED retail_demo.silver.promotion_sales;

In [0]:
%sql
-- Adding comments to retail_demo.gold.promotion_impact
COMMENT ON COLUMN retail_demo.gold.promotion_impact.product_id IS 'Identifier for the product analyzed';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.product_name IS 'Human-readable product name';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.category IS 'Product category';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.promotion_id IS 'Identifier for the promotion analyzed';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.promotion_name IS 'Human-readable promotion name';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.promotion IS 'Flag indicating whether a promotion was applied';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.regular_price IS 'Standard price before promotion';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.discount_pct IS 'Percentage discount applied (0-100)';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.gross_margin_pct IS 'Gross margin percentage (0-100)';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.baseline_units IS 'Estimated units that would have been sold without the promotion';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.baseline_revenue IS 'Estimated revenue without the promotion';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.promotion_units IS 'Actual units sold during the promotion period';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.promotion_revenue IS 'Actual revenue during the promotion period';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.net_selling_price IS 'Effective selling price after discount';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.promotion_investment IS 'Total cost of the promotion (discount amount given up)';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.incremental_units IS 'Additional units sold due to the promotion (promotion - baseline)';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.incremental_revenue IS 'Additional revenue generated by the promotion';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.incremental_gross_profit IS 'Additional gross profit from the promotion';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.revenue_uplift_pct IS 'Percentage revenue increase vs baseline (already multiplied by 100)';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.unit_uplift_pct IS 'Percentage unit increase vs baseline (already multiplied by 100)';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.promotion_effectiveness IS 'Qualitative rating: Effective, Moderate, or Ineffective';
COMMENT ON COLUMN retail_demo.gold.promotion_impact.gold_ingested_at IS 'Timestamp when this row was loaded into the gold layer';

-- Verify
DESCRIBE TABLE EXTENDED retail_demo.gold.promotion_impact;

## PII Tagging & Data Governance

Unity Catalog supports tags for data classification. We tag PII columns on the bronze customer table and then apply column masks to protect sensitive data.

The `retail_demo.bronze.customer` table has columns:
- `Id` (INT) — customer identifier (tag as sensitive)
- `name` (STRING) — customer name, PII data (tag as pii, apply mask)
- `email` (STRING) — customer email, PII data (tag as pii, apply mask)
- `_created_at`, `_updated_at` (TIMESTAMP) — metadata timestamps

In [0]:
%sql
-- Apply data classification tags to the bronze customer table
-- Unity Catalog auto-creates tags when first applied (no CREATE TAG needed)
-- Syntax: ALTER TABLE ... ALTER COLUMN ... SET TAGS ('key' = 'value')

-- Tag the name column as PII (customer name is personally identifiable)
ALTER TABLE retail_demo.bronze.customer
  ALTER COLUMN name SET TAGS ('pii' = 'true');

-- Tag the email column as PII (email is personally identifiable)
ALTER TABLE retail_demo.bronze.customer
  ALTER COLUMN email SET TAGS ('pii' = 'true');

-- Tag the Id column as sensitive (customer identifier)
ALTER TABLE retail_demo.bronze.customer
  ALTER COLUMN Id SET TAGS ('sensitive' = 'true');

-- Verify the tags via information schema
SELECT column_name, tag_name, tag_value
FROM retail_demo.information_schema.column_tags
WHERE table_name = 'customer'
ORDER BY column_name, tag_name;

In [0]:
%sql
-- Column masks dynamically redact sensitive data based on who is querying
-- Non-privileged users see masked values; authorized users see the real data

-- Create a masking function for the name column (show first character + ***)
CREATE OR REPLACE FUNCTION retail_demo.bronze.mask_pii(value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_member('retail_demo_admins') THEN value
    ELSE CONCAT(SUBSTRING(value, 1, 1), '***')
  END;

-- Create a masking function for the email column (mask username, show domain)
CREATE OR REPLACE FUNCTION retail_demo.bronze.mask_email(email_val STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_member('retail_demo_admins') THEN email_val
    ELSE CONCAT('***', SUBSTRING(email_val, INSTR(email_val, '@')))
  END;

-- Apply the masks to the customer table
ALTER TABLE retail_demo.bronze.customer
  ALTER COLUMN name SET MASK retail_demo.bronze.mask_pii;

ALTER TABLE retail_demo.bronze.customer
  ALTER COLUMN email SET MASK retail_demo.bronze.mask_email;

-- Query the table — if you are NOT in the retail_demo_admins group, you will see masked values
-- name shows as "J***", email shows as "***@gmail.com"
SELECT * FROM retail_demo.bronze.customer LIMIT 10;

## Summary

- **Namespace**: catalog.schema.table 3-level hierarchy
- **Comments**: Self-documenting columns and tables
- **PII Tags**: Tags classify sensitive data (pii, sensitive)
- **Column Masks**: Dynamic redaction based on group membership
- **Lineage**: Bronze → Silver → Gold data flow
- **Discovery**: Information schema queries for auditing and compliance

To see the visual lineage graph, open any table in Catalog Explorer and click the Lineage tab.